Prepare data

In [1]:
import torch
import mlrun

from dotenv import load_dotenv
load_dotenv() 

from pathlib import Path
from datetime import datetime

artifact_path = Path.cwd().parent
artifact_path = str(artifact_path.as_posix()) # convert windows path to unix path
artifact_path = "file://" + artifact_path
p = mlrun.set_environment("http://localhost:8080", artifact_path=artifact_path)

project = mlrun.load_project(name='finetune-legal-extractor', context="../") # project 

In [5]:
prompt_artifact = project.get_artifact(key="contract_extractor_prompt", tag="20260610_1257")
sys_prompt = prompt_artifact.read_prompt()[0]['content']
invoc_config = prompt_artifact.to_dict()['spec']['invocation_config']
invoc_config['max_new_tokens'] = 2000 # hard cap

In [17]:
import pyarrow.dataset as ds
artifact_latest_s3_path = project.get_artifact(key="raw-proc-process-raw_test_data", tag="20260506_1224").target_path

test_dataset = ds.dataset(
        source=artifact_latest_s3_path, 
        format="parquet")
dataset_list = test_dataset.to_table().to_pylist()[:10] # batch size is about 2, send 20 concurrent requests

In [18]:
dataset_list = [x['text'] for x in dataset_list]

In [ ]:
echo '{"inputs":"The new movie that got Oscar this year","parameters":{"max_new_tokens":256, "do_sample":true}}' > prompts/prompt1.txt

In [20]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("JerroldK/Hermes-4-14B-contract-extractor")
import json

i = 1
for contract in dataset_list:
    full_messages = [
                    {"role": "system", "content": sys_prompt},
                    {"role": "user", "content": contract},
                ]
    # Create a text file for each prompt, prompt is str
    prompt = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    payload = {
        "inputs": prompt,
        "parameters": invoc_config
    }
    filename = f"prompts/prompt{i}.txt"
    with open(filename, 'w', encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    print(f'Wrote {filename}')

    i += 1

Wrote prompts/prompt1.txt
Wrote prompts/prompt2.txt
Wrote prompts/prompt3.txt
Wrote prompts/prompt4.txt
Wrote prompts/prompt5.txt
Wrote prompts/prompt6.txt
Wrote prompts/prompt7.txt
Wrote prompts/prompt8.txt
Wrote prompts/prompt9.txt
Wrote prompts/prompt10.txt


In [14]:
dataset_list[0].keys()

dict_keys(['id', 'document_id', 'text', 'inference', 'model_repo', 'model_repo_version', 'timestamp', 'origin', '__index_level_0__'])